In [13]:
'''
Created by Zhaogang Huang @ Nov 9th, 2024
--------------------
This file aims to extract CSMAR/CNRDS data from original data
files. Additionally, the file creates a dictionary mapping 
each variable name to its corresponding definition, ensuring
clarity and ease of reference.
'''
import os
import re
import pandas as pd

In [14]:
def columns_name(loc):
    file_name: str
    for s in os.listdir(loc):
        if s.find('txt') != -1 and s[:2] != '._':
            file_name = s
            break
    
    with open(loc+'/'+file_name, 'r', encoding='utf-8-sig') as f:
        columns = [i.split('-')[0].strip() for i in f.readlines()]
        columns = [i for i in columns if i != '']
        col1 = [i.split('[')[0].strip() for i in columns]
        col2 = [i.split('[')[1].replace(']', '').strip() for i in columns]
    
    return col1, col2

def combile_data(loc):
    '''
    This function is to combine all data in the same folder.
    Only support ``.xlsx'' and ``.csv''
    
    input
    --------------------
    loc: the location of the data file

    output
    --------------------
    df: combined data
    col_dict: the dictionary mapping variable name to its definition
    '''
    file_type = [s.split('.')[-1] for s in os.listdir(loc) if (s.split('.')[-1] in ['csv',  'xlsx'] and s[:2] != '._')]
    file_type = '.'+file_type[0]
    file_list = [s for s in os.listdir(loc) if s.find(file_type) != -1 and s[:2] != '._']
    col1, col2 = columns_name(loc)

    df = pd.DataFrame(columns=col1)
    for file in file_list:
        if file_type == '.csv':
            d = pd.read_csv(loc  + file)
        elif file_type == '.xlsx':
            d = pd.read_excel(loc  + file, engine='openpyxl')
            d.drop(index=[0, 1], inplace=True)

        df = pd.concat([df, d]).reset_index(drop=True)

    col_dict = {col1[i]:col2[i] for i in range(len(col1))}
    return df, col_dict

def convert_to_parquet(loc, save_link, spcific_type=[]):
    '''
    This function is to convert all data in the file named ``CSMAR'' 
    or ``CNRDS'' into parquet files

    input
    --------------------
    loc: the location of CSMAR/CNRDS data
    save_link: the place where you want to save the converted data
    spcific_type: the spcific data that need to be convert

    output
    --------------------
    None
    '''

    pattern = r'[\u4e00-\u9fff]+' # All Chinese Characters
    files = list(set([re.findall(pattern, s)[0] for s in os.listdir(loc) if re.findall]))

    for ftype in files:
        if len(spcific_type) > 0 and (ftype not in spcific_type):
            continue

        print('now is proceeding with {0}'.format(ftype))
        file_list = [s for s in os.listdir(loc) if s.find(ftype) != -1 and s[:2] != '._']
        dfs = []
        for f in file_list:
            df, col_dict = combile_data(loc+f+'/')
            dfs.append(df)
        df_combined = pd.concat(dfs, axis=0)
        df_combined.to_parquet(save_link + 'CSMAR_'+ftype+'.parquet')

In [15]:
folder = '../'
convert_to_parquet(
    folder + '03_data/financial_statement/original/', 
    folder + '03_data/financial_statement/', 
    ['上市公司基本信息年度表']
)

now is proceeding with 上市公司基本信息年度表


d:\Python313\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
